# AmsiNet - Step 24: confusion-matrix figure for Section 5.1 (CPU, about 1 minute)
Draws the row-normalised confusion matrices on the **test words** (five seeds pooled) for the EfficientNet-B0 word agent (AmsiNet-E, alone) and the flat ViT-B/16, side by side, and saves `figures/FigC_confusion_matrices.png`. Run all, then send me the PNG (download it from Drive or from the Colab file panel).

In [ ]:
from google.colab import drive
import os, json, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Serif'
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/amsinet'; os.makedirs(f'{ROOT}/figures', exist_ok=True)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SC = D['scripts']; SI = {s: i for i, s in enumerate(SC)}; C = len(SC)
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv')
AB = {'Bengali-Assamese': 'Beng.-Assam.', 'Devanagari': 'Devanagari', 'Gujarati': 'Gujarati', 'Gurmukhi': 'Gurmukhi', 'Kannada': 'Kannada', 'Latin': 'Latin', 'Malayalam': 'Malayalam', 'Odia': 'Odia', 'Tamil': 'Tamil', 'Telugu': 'Telugu'}
def conf(tag, sp='test'):
    d = META[META.split == sp].reset_index(drop=True); y = np.array([SI[s] for s in d.script]); M = np.zeros((C, C))
    for s in range(5):
        P = np.load(f'{ROOT}/agents/{tag}_seed{s}_{sp}_probs.npy'); assert pd.read_csv(f'{ROOT}/agents/{tag}_seed{s}_{sp}_paths.csv').path.tolist() == d.path.tolist()
        np.add.at(M, (y, P.argmax(1)), 1)
    return 100 * M / M.sum(1, keepdims=True)
MS = [('AmsiNet-E word agent (EfficientNet-B0), alone', conf('flat_effb0')), ('Flat ViT-B/16', conf('flatvit'))]
fig, axs = plt.subplots(1, 2, figsize=(13, 5.8), gridspec_kw={'wspace': 0.55})
for ax, (title, M) in zip(axs, MS):
    im = ax.imshow(M, cmap='Blues', vmin=0, vmax=100)
    for i in range(C):
        for j in range(C):
            v = M[i, j]
            if i == j or v >= 1.0: ax.text(j, i, f'{v:.0f}' if i == j else f'{v:.1f}', ha='center', va='center', fontsize=7.5, color='white' if v > 55 else 'black')
    ax.set_xticks(range(C)); ax.set_yticks(range(C)); ax.set_xticklabels([AB[s] for s in SC], rotation=45, ha='right', fontsize=8); ax.set_yticklabels([AB[s] for s in SC], fontsize=8)
    ax.set_xlabel('Predicted script'); ax.set_title(title, fontsize=9.5)
    if ax is axs[0]: ax.set_ylabel('True script')
fig.colorbar(im, ax=axs, fraction=0.025, pad=0.02, label='% of the true class')
fig.savefig(f'{ROOT}/figures/FigC_confusion_matrices.png', dpi=300, bbox_inches='tight'); plt.show()
print('saved', f'{ROOT}/figures/FigC_confusion_matrices.png')